# Diffusion-model arithmetic: one forward noise step

Companion to HTB Academy Module 290, Section 23. This notebook traces **one chosen noise draw** and an oracle algebra check. It does **not** train a denoiser, implement a full diffusion sampler, or generate an image.

Read xₜ=√(1−βₜ)xₜ₋₁+√βₜ ε as **x sub t equals square root of one minus beta sub t times x sub t-minus-one, plus square root of beta sub t times epsilon**. The first term keeps earlier signal; the second adds one random-noise draw.

In [ ]:
import math

previous = (1.0, -1.0)  # A two-component toy sample x_(t-1).
noise = (0.0, 2.0)  # One deliberately chosen epsilon draw, not a dataset.
beta = 0.25  # Noise variance at this toy time step.
signal_scale = math.sqrt(1.0 - beta)
noise_scale = math.sqrt(beta)
noisy = tuple(signal_scale * old + noise_scale * eps
              for old, eps in zip(previous, noise))
for index, (old, eps, result) in enumerate(zip(previous, noise, noisy)):
    print(f'component {index}: {signal_scale:.3f} * {old:+.1f} + {noise_scale:.3f} * {eps:+.1f} = {result:+.3f}')
assert abs(noisy[0] - math.sqrt(0.75)) < 1e-12
assert abs(noisy[1] - (1 - math.sqrt(0.75))) < 1e-12

## Compare actual and predicted noise

The training objective compares the true added noise ε (“epsilon”) with ε-pred, the model's guess. For our invented guess (0,1.5), the squared error is 0.25. The course's expected loss E[||ε−ε-pred||²], read **expected squared norm of noise minus predicted noise**, averages such errors across many draws. Using the *true* noise below can invert this one algebraic step, but a real image generator does not possess that oracle knowledge.

In [ ]:
predicted_noise = (0.0, 1.5)
squared_error = sum((actual - predicted) ** 2
                    for actual, predicted in zip(noise, predicted_noise))
oracle_reconstruction = tuple((current - noise_scale * actual) / signal_scale
                              for current, actual in zip(noisy, noise))
print('one-example squared noise error:', squared_error)
print('oracle reconstruction with true noise:', oracle_reconstruction)
assert squared_error == 0.25
assert all(abs(recovered - original) < 1e-12
           for recovered, original in zip(oracle_reconstruction, previous))

## Check the course's linear beta schedule

Read βₜ=β-min+(t/T)(β-max−β-min) as **beta sub t equals beta minimum plus t over T times beta maximum minus beta minimum**. With t=0 through T, this interpolation includes both endpoints. The schedule changes the noise level; it does not by itself learn denoising.

In [ ]:
beta_min, beta_max, total_steps = 0.1, 0.3, 4
schedule = [beta_min + (step / total_steps) * (beta_max - beta_min)
            for step in range(total_steps + 1)]
print('linear beta schedule:', schedule)
assert abs(schedule[0] - beta_min) < 1e-12
assert abs(schedule[-1] - beta_max) < 1e-12
assert all(0 < value < 1 for value in schedule)